# 170. SPLADE：学习型稀疏扩展、FLOPS 正则与倒排检索怎样实现？

> **面试问题：SPLADE 与 BM25/稠密检索有什么区别？`log(1+ReLU)`、max pooling、稀疏正则和 impact index 怎样落地？**

## 先给结论

SPLADE 用语言模型词表维度表示 query/document，每个维度仍可进入倒排索引，但非原文词也可获得权重形成语义扩展。常见聚合是对 token 位置的 `log(1+ReLU(logit))` 取 max；训练以排序/蒸馏质量和稀疏正则权衡，线上最终成本由非零维度与 postings 分布决定。

## 推荐回答主线

1. 区分输入 token 与输出词表维度，实现非负饱和变换和 position max pooling。
2. 证明稀疏向量点积可由倒排 postings 累加，解释 learned expansion 如何召回非字面匹配。
3. 实现 ranking/distillation loss 与 FLOPS-style regularizer，并观察稀疏—质量权衡。
4. 覆盖 top-k/阈值、impact 量化、长 posting 热点、ACL、索引版本与 nDCG/延迟评测。

## 教学边界

代码用小型 Embedding+Linear 产生词表 logits，倒排索引是内存字典；不复现预训练 MLM、真实 SPLADE checkpoint、Lucene impact index 或分布式查询执行。

## 一手资料

- [SPLADE](https://arxiv.org/abs/2107.05720)
- [SPLADE v2](https://arxiv.org/abs/2109.10086)
- [Efficient SPLADE](https://arxiv.org/abs/2207.03834)


In [ ]:
import hashlib
import json
import math
from collections import defaultdict
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

# 0 是 padding，输出轴是完整词表而非输入序列长度。
torch.manual_seed(170)
VOCAB, DIM = 20, 10
documents = torch.tensor([[1, 2, 3, 0], [4, 5, 0, 0], [1, 6, 7, 0]])
queries = torch.tensor([[1, 8, 0], [4, 9, 0]])
d_mask, q_mask = documents.ne(0), queries.ne(0)

assert documents.max().item() < VOCAB
assert d_mask.shape == documents.shape
assert q_mask.sum().item() == 4


## 1. 手写 SPLADE 表示：非负变换后沿 token 位置取 max

每个输入位置输出 V 维词表 logits，先做 `log1p(ReLU)` 抑制极大权重，再在有效位置取 max。padding 位置要在聚合前归零；空文本应拒绝或返回全零，避免 max 的无定义语义。


In [ ]:
class TinySPLADE(nn.Module):
    def __init__(self, vocab, dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab, dim, padding_idx=0)
        self.mlm_head = nn.Linear(dim, vocab)

    def forward(self, token_ids, mask):
        logits = self.mlm_head(self.embedding(token_ids))
        impacts = torch.log1p(torch.relu(logits)) * mask[..., None]
        return impacts.max(dim=1).values, impacts

# 输出维度等于词表；权重非负；padding 位置不贡献 impact。
model = TinySPLADE(VOCAB, DIM)
d_sparse, d_position = model(documents, d_mask)
q_sparse, q_position = model(queries, q_mask)
assert d_sparse.shape == (3, VOCAB)
assert (d_sparse >= 0).all()
assert d_position[~d_mask].abs().sum().item() == 0.0


## 2. 学习型扩展：输出词不必在原文出现

MLM head 可以给未出现在文档中的相关词非零权重，这就是 expansion。下面手工 logits 隔离该性质：文档只含 token 1，却在输出词 8 上激活，从而可与只含词 8 的 query 匹配。扩展也会引入漂移和长 postings。


In [ ]:
def splade_pool(logits, mask):
    transformed = torch.log1p(torch.relu(logits)) * mask[..., None]
    return transformed.max(1).values

# 文档输入没有词 8，但输出词表维度 8 可被模型扩展激活。
manual_logits = torch.full((1, 2, VOCAB), -2.0)
manual_logits[0, 0, 1] = 3.0
manual_logits[0, 0, 8] = 2.0
manual_repr = splade_pool(manual_logits, torch.tensor([[1, 0]], dtype=torch.bool))
assert manual_repr[0, 8] > 0
assert 8 not in [1]
assert manual_repr[0, 9] == 0


## 3. 倒排 index：稀疏点积等于 postings impact 累加

对每个非零词表维度保存 `(doc_id, impact)`，查询只遍历自己的非零维度并累加乘积。数学上等于 dense dot product；工程上成本取决于 query 非零数与每个 term 的 posting 长度。


In [ ]:
def build_impact_index(document_vectors, threshold=0.0):
    index = defaultdict(list)
    for doc_id, vector in enumerate(document_vectors):
        for term in torch.nonzero(vector > threshold, as_tuple=False).flatten().tolist():
            index[term].append((doc_id, float(vector[term])))
    return dict(index)

def sparse_search(query_vector, index, n_docs, threshold=0.0):
    scores = np.zeros(n_docs, dtype=float)
    for term in torch.nonzero(query_vector > threshold, as_tuple=False).flatten().tolist():
        for doc_id, impact in index.get(term, []):
            scores[doc_id] += float(query_vector[term]) * impact
    return scores

# 倒排累加与完整矩阵乘一致，未知/零 query 返回全零。
impact_index = build_impact_index(d_sparse.detach())
inverted_scores = sparse_search(q_sparse[0].detach(), impact_index, len(documents))
dense_scores = (q_sparse[0].detach() @ d_sparse.detach().T).numpy()
assert np.allclose(inverted_scores, dense_scores)
assert sparse_search(torch.zeros(VOCAB), impact_index, len(documents)).sum() == 0
assert all(term < VOCAB for term in impact_index)


## 4. 排序与蒸馏损失：学习相对相关性，不把 teacher 当真理

可对正负文档做 pairwise softplus，也可拟合 cross-encoder teacher 的软分布。teacher 会携带偏差，训练/验证必须按 query family 切分，并检查 false negative。下面组合 pairwise 与 KL 蒸馏。


In [ ]:
def ranking_distillation_loss(query_vectors, document_vectors, positive_ids, teacher_logits, alpha=0.6):
    student = query_vectors @ document_vectors.T
    positive = student[torch.arange(len(query_vectors)), positive_ids]
    hardest_negative = student.masked_fill(F.one_hot(positive_ids, student.shape[1]).bool(), -torch.inf).max(1).values
    pairwise = F.softplus(hardest_negative - positive).mean()
    distill = F.kl_div(F.log_softmax(student, -1), F.softmax(teacher_logits, -1), reduction="batchmean")
    return alpha * pairwise + (1 - alpha) * distill, student

# loss 有限并能更新 encoder；正例 id 落在文档范围内。
positive_ids = torch.tensor([0, 1])
teacher = torch.tensor([[3.0, 0.5, -1.0], [-0.5, 2.5, 0.0]])
rank_loss, student_scores = ranking_distillation_loss(q_sparse, d_sparse, positive_ids, teacher)
model.zero_grad(set_to_none=True); rank_loss.backward(retain_graph=True)
assert torch.isfinite(rank_loss)
assert positive_ids.max().item() < len(documents)
assert model.mlm_head.weight.grad is not None


## 5. FLOPS-style 正则：惩罚 batch 平均激活的平方

常用 surrogate 对每个词表维度的 batch 平均权重平方求和；高频激活维度惩罚更大，间接控制 postings。query/document 可用不同系数。它不是硬 FLOP 计数，真实延迟仍取决于分布和引擎。


In [ ]:
def flops_regularizer(sparse_vectors):
    mean_activation = sparse_vectors.mean(dim=0)
    return (mean_activation ** 2).sum()

# 加倍所有 impact 会让正则放大四倍；全零表示正则为零。
doc_reg = flops_regularizer(d_sparse)
query_reg = flops_regularizer(q_sparse)
assert doc_reg >= 0 and query_reg >= 0
assert torch.allclose(flops_regularizer(2 * d_sparse), 4 * doc_reg)
assert flops_regularizer(torch.zeros_like(d_sparse)).item() == 0.0


## 6. 剪枝与 impact 量化：减少 postings，同时监控排序翻转

部署可按 top-k 或阈值剪掉小权重，再把 impact 映射到整数。top-k 给固定每文档上限，阈值更贴合绝对权重但文档间长度不同。量化 scale 必须随索引版本保存。


In [ ]:
def topk_prune(vectors, k):
    values, ids = torch.topk(vectors, min(k, vectors.shape[1]), dim=1)
    pruned = torch.zeros_like(vectors)
    pruned.scatter_(1, ids, values)
    return pruned

def quantize_impacts(vectors, levels=255):
    scale = vectors.max().clamp_min(1e-8) / levels
    quantized = torch.round(vectors / scale).clamp(0, levels).to(torch.uint8)
    return quantized, scale

# top-k 控制每行非零上限；反量化误差不超过半个 scale（含浮点余量）。
pruned = topk_prune(d_sparse.detach(), k=5)
impact_q, impact_scale = quantize_impacts(pruned)
restored = impact_q.float() * impact_scale
assert (pruned > 0).sum(1).max().item() <= 5
assert (restored - pruned).abs().max() <= impact_scale / 2 + 1e-6
assert impact_q.dtype == torch.uint8


## 7. 效率指标：平均非零数不够，还要看 posting 长尾

少数扩展词若出现在大部分文档，会成为热 posting。报告 document/query 非零数分位、每 term document frequency、每查询访问 postings、压缩字节和 p50/p99；质量则用 Recall/nDCG/MRR 与 lexical/dense baseline 对照。


In [ ]:
def sparsity_report(vectors, threshold=0.0):
    active = vectors > threshold
    nnz_per_doc = active.sum(1).cpu().numpy()
    df = active.sum(0).cpu().numpy()
    return {
        "mean_nnz": float(nnz_per_doc.mean()),
        "max_nnz": int(nnz_per_doc.max()),
        "max_df": int(df.max()),
        "total_postings": int(active.sum()),
    }

# 剪枝后总 postings 不增，每词 df 不超过文档数，统计量非负。
before = sparsity_report(d_sparse.detach())
after = sparsity_report(pruned)
assert after["total_postings"] <= before["total_postings"]
assert after["max_df"] <= len(documents)
assert after["mean_nnz"] >= 0


## 8. 索引发布：模型与 analyzer/词表/剪枝 recipe 原子绑定

SPLADE 使用固定 tokenizer 词表维度，词表 id 错一位就会查询错误 posting。manifest 还应绑定模型、最大长度、剪枝、impact scale、文档快照和 ACL；迁移期双建索引并做 shadow query。


In [ ]:
@dataclass(frozen=True)
class SparseIndexManifest:
    model_hash: str
    tokenizer_hash: str
    vocab_size: int
    prune_k: int
    document_snapshot: str
    acl_snapshot: str

def manifest_digest(manifest):
    return hashlib.sha256(json.dumps(asdict(manifest), sort_keys=True).encode()).hexdigest()

# 在线 query 维度必须匹配 manifest，任何剪枝 recipe 变化都生成新制品。
manifest = SparseIndexManifest("splade-demo-v2", "tok-v5", VOCAB, 5, "docs-42", "acl-9")
digest = manifest_digest(manifest)
assert manifest.vocab_size == q_sparse.shape[1]
assert len(digest) == 64
assert digest != manifest_digest(SparseIndexManifest("splade-demo-v2", "tok-v5", VOCAB, 8, "docs-42", "acl-9"))


## 面试收束与生产替换点

完整回答不要停在算法名：先说清业务目标、输入输出与信任边界，再给核心数据结构/公式和可执行 oracle，最后落到离线切片、线上 SLO、成本、安全、版本、灰度与回滚。这里的受控实现用于解释机制和发现反例；真实模型编码器、分布式索引、协议 SDK、安全沙箱、监控与持久层应作为可替换组件，并用同一合同验收。

典型追问包括：数据规模扩大后瓶颈在哪？近似步骤损失了什么？哪个状态必须持久化？超时或部分失败怎样降级？版本错配为何不能静默兼容？离线指标上升是否来自污染、权限泄漏、评测器偏差或重复样本？
